# Introduction to the Memory Management Interface

## 1. Batched matrix-vector multiplication

The operation computes `result = features @ weights.T`, i.e.:

$$\text{output}[i, j] = \sum_k \text{features}[i, k] \cdot \text{weights}[j, k]$$

The Q.ANT SDK provides three interfaces for this operation, which give progressively more control over device memory.

## Setup

In [ ]:
import numpy as np
import qant_native_computing_toolkit.ai as q_ai
import qant_native_computing_toolkit.generic as q_generic
from ml_dtypes import bfloat16

We fix the random seed, create random test data and define a helper that compares a result against a float32 NumPy reference.

In [ ]:
np.random.seed(0)

n_batches, in_features, out_features = 32, 128, 64


def random_bf16(*shape):
    return np.random.uniform(-1.0, 1.0, size=shape).astype(bfloat16)


features = random_bf16(n_batches, in_features)
features2 = random_bf16(n_batches, in_features)
weights = random_bf16(out_features, in_features)

### Level 1: No explicit memory management

`linear_fprop` works directly on NumPy arrays and implicitly allocates, uploads, downloads and frees the device memory it needs. This interface does not allow any reuse and is therefore the slowest option for repeated calls.

In [ ]:
results = q_ai.linear_fprop(features, weights)

### Level 2: Reuse the weights

`linear_fprop_preloaded_weights` expects the weights to be uploaded beforehand with `upload` or `upload_into` (with `data_role=DataRole.Weights`). The weights can then stay on the device and be reused for repeated calculations. The features and results are still allocated, uploaded, downloaded and freed implicitly.

In [ ]:
weights_handle = q_generic.upload(weights, data_role=q_generic.DataRole.Weights)

results = q_ai.linear_fprop_preloaded_weights(features, weights_handle)
results2 = q_ai.linear_fprop_preloaded_weights(features2, weights_handle)

### Level 3: Full memory management

`linear_fprop_preloaded` expects both the weights (with `data_role=DataRole.Weights`) and the features (with `data_role=DataRole.Features`) to be uploaded beforehand. Allocating the result buffer is optional: if no `result_handle` is passed, a new one is allocated and returned. Unlike the previous levels, the function returns a handle rather than an array, so the result has to be downloaded explicitly.

Since every handle can be reused, this saves one or two allocations per call compared to Level 2.

Additionally, with explicit results handles, consecutive operations can be chained without transferring the intermediate result to the device.

In [ ]:
# upload the features to the device
features_handle = q_generic.upload(features, data_role=q_generic.DataRole.Features)

# allocate the result buffer explicitly
results_handle = q_generic.allocate(
    n_batches, out_features, data_role=q_generic.DataRole.Features
)

# execute the multiplication on the device and download the result
q_ai.linear_fprop_preloaded(features_handle, weights_handle, results_handle)
results = q_generic.download(results_handle)

# reuse all handles for the next batch: only new data is uploaded, nothing is allocated
q_generic.upload_into(features2, features_handle)
q_ai.linear_fprop_preloaded(features_handle, weights_handle, results_handle)
results2 = q_generic.download(results_handle)

# reuse the results for an additional multiplication
weights2 = random_bf16(out_features, out_features)
weights_handle2 = q_generic.upload(weights2, data_role=q_generic.DataRole.Weights)
results_handle3 = q_ai.linear_fprop_preloaded(results_handle, weights_handle2)
results3 = q_generic.download(results_handle3)

### Cleanup

Device memory is released automatically once a handle is garbage collected. To release it explicitly, delete the handles.

In [ ]:
del weights_handle, features_handle, results_handle

## 2. Convolution

For the convolution, only the first two levels are available. The transposed convolution works the same way with `conv_transpose_fprop`, `upload_conv_transpose_weights` and `conv_transpose_fprop_preloaded_weights`.

### Setup

In [ ]:
n_batches, in_channels, in_height, in_width = 8, 3, 32, 32
out_channels, kernel_height, kernel_width = 16, 3, 3
padding, stride, dilation = 0, 1, 1
features = random_bf16(n_batches, in_channels, in_height, in_width)
features2 = random_bf16(n_batches, in_channels, in_height, in_width)
weights = random_bf16(out_channels, in_channels, kernel_height, kernel_width)

### Level 1: No explicit memory management

`conv_fprop` works directly on NumPy arrays and implicitly allocates, uploads, downloads and frees the device memory it needs. This interface does not allow any reuse and is therefore the slowest option for repeated calls.

In [ ]:
results = q_ai.conv_fprop(
    features, weights, padding=padding, stride=stride, dilation=dilation
)

### Level 2: Reuse the weights

`conv_fprop_preloaded_weights` expects the weights to be uploaded beforehand with `upload_conv_weights`. The weights can then stay on the device and be reused for repeated calculations. The features and results are still allocated, uploaded, downloaded and freed implicitly.

In [ ]:
weights_handle = q_ai.upload_conv_weights(weights)

results = q_ai.conv_fprop_preloaded_weights(
    features, weights_handle, padding=padding, stride=stride, dilation=dilation
)
results2 = q_ai.conv_fprop_preloaded_weights(
    features2, weights_handle, padding=padding, stride=stride, dilation=dilation
)

### Why is there no Level 3 for the convolution?

Level 3 helps because the features and results stay on the device and their buffers are reused across calls. That doesn't carry over well to the convolution: the buffer sizes depend on the input image size, which can be anything, and the data is reordered before and after the computation, so the buffers can't be reused directly as in `linear_fprop`. A Level 3 convolution would only save the two allocations for features and results, and only when the image size, padding, stride and dilation stay the same between calls. That benefit is too small to justify another API layer with stricter usage constraints, so we decided not to offer a Level 3 convolution.

If you need this level of control over device memory, you can build your own convolution on top of `linear_fprop_preloaded`, for example by unfolding the input patches into a matrix (im2col) and multiplying it with the flattened kernel.

### Cleanup

As in the first example, the weights handle is released automatically once it is garbage collected. To release it explicitly, delete it with `del weights_handle`.